# 🖥️ Computing derived fields with xgcm

Sometimes the field you want particles to sample isn't in the model output directly, but can be derived from it - for example the relative vorticity, divergence or strain rate of the flow. On a staggered (C-)grid, computing such derivatives correctly requires knowing exactly where each variable lives on the grid.

Because Parcels ingests structured grid data via [SGRID conventions](https://sgrid.github.io/sgrid/), that grid information is already attached to the dataset. In this tutorial we will:

1. Construct an SGRID dataset from MITgcm output
2. Set up an [xgcm](https://xgcm.readthedocs.io/) `Grid` directly from the SGRID metadata
3. Use xgcm to compute the relative vorticity of the flow, and plot it
4. Run a Parcels simulation in which particles sample the vorticity along their trajectories

In [ ]:
import cmocean
import matplotlib.pyplot as plt
import numpy as np
import xgcm

import parcels
import parcels.tutorial

print(f"xgcm version: {xgcm.__version__}")

## Constructing the SGRID dataset

We use the surface velocities from an idealised, zonally re-entrant MITgcm channel simulation (see also the [MITgcm tutorial](tutorial_mitgcm.ipynb)). The grid is a Cartesian C-grid with 5 km spacing.

In [ ]:
ds_fields = parcels.tutorial.open_dataset(
    "MITgcm_example_data/mitgcm_UV_surface_zonally_reentrant"
)

{py:func}`parcels.convert.mitgcm_to_sgrid` converts the MITgcm output into an SGRID-compliant dataset. Along with the velocities, we also want to keep the grid metrics that we need for computing the vorticity: the distances between the velocity points (`dxC` and `dyC`) and the area of the cells around the cell corners (`rAz`). We attach these as coordinates (so that Parcels doesn't treat them as fields), renaming their dimensions to match those used in the SGRID dataset.

In [ ]:
ds_fset = parcels.convert.mitgcm_to_sgrid(
    fields={"U": ds_fields.UVEL, "V": ds_fields.VVEL}, coords=ds_fields
)

# mitgcm_to_sgrid only keeps the grid axes (XG, YG, Zl) from `coords` and drops all
# other variables there, including the grid metrics.
# Let's add back the grid metrics
ds_fset = ds_fset.assign_coords(
    dxC=ds_fields.dxC.rename(XG="lon"),
    dyC=ds_fields.dyC.rename(YG="lat"),
    rAz=ds_fields.rAz.rename(XG="lon", YG="lat"),
)
ds_fset

The SGRID metadata, stored in the attributes of the `grid` variable, describes how the dimensions of the dataset relate to each other. We can inspect it with the `sgrid` accessor:

In [ ]:
ds_fset.sgrid.metadata.describe()

For each horizontal axis, the metadata pairs a _face_ dimension (the cell centres, `XC` and `YC`) with a _node_ dimension (the cell corners, `lon` and `lat`), together with a _padding_ that describes how the two are offset from each other. Here `padding=high` means that there is a face beyond the last node, i.e., each node sits on the "left" of the face with the same index.

## Setting up the xgcm Grid

xgcm can parse SGRID metadata natively, translating the SGRID padding of each axis into the corresponding xgcm position (e.g., `padding=high` means the nodes are on the `left` of the faces). There are two small things we need to do before handing the dataset over to xgcm:

1. xgcm only looks for SGRID metadata if the dataset declares it follows SGRID conventions in its global `Conventions` attribute.
2. The MITgcm conversion describes the vertical axis using the same `depth` dimension for both faces and nodes, which xgcm can't represent. As this dataset only contains surface velocities we don't need the vertical axis, so we remove `vertical_dimensions` from the metadata.

We make these changes on a copy of the dataset, so that the dataset we pass to Parcels is left untouched.

In [ ]:
ds_xgcm = ds_fset.copy()
ds_xgcm.attrs["Conventions"] = "SGRID"
del ds_xgcm["grid"].attrs["vertical_dimensions"]

The channel is periodic in the zonal (X) direction and bounded by walls in the meridional (Y) direction, so we use periodic padding in X and fill with zeros (no flow through the walls) in Y. The axes and positions of the dimensions are all parsed from the SGRID metadata.

In [ ]:
grid = xgcm.Grid(
    ds_xgcm,
    padding={"X": "periodic", "Y": "fill"},
    fill_value={"Y": 0.0},
)
grid

## Computing the vorticity

The relative vorticity $\zeta = \partial v / \partial x - \partial u / \partial y$ is naturally defined on the cell corners of a C-grid. Using the finite-volume formulation (as used in MITgcm),

$$
\zeta = \frac{\delta_x (v \, \Delta y_C) - \delta_y (u \, \Delta x_C)}{A_\zeta},
$$

where $\delta_x$ and $\delta_y$ are differences between neighbouring points, which is what {py:meth}`xgcm.Grid.diff` computes. Differencing `V` (on `(lat, XC)`) in X and `U` (on `(YC, lon)`) in Y both bring the result to the cell corners `(lat, lon)`.

In [ ]:
vorticity = (
    grid.diff(ds_fset.V * ds_fset.dyC, "X") - grid.diff(ds_fset.U * ds_fset.dxC, "Y")
) / ds_fset.rAz
vorticity.attrs = {"long_name": "Relative vorticity", "units": "s-1"}
vorticity.dims

Let's plot the vorticity at the first time step to check that it looks sensible. Since vorticity can be positive (anticlockwise) or negative (clockwise), we use a diverging colourmap centred on zero.

In [ ]:
vmax = 2e-5
fig, ax = plt.subplots(figsize=(5, 8))
pc = ax.pcolormesh(
    vorticity.lon / 1e3,
    vorticity.lat / 1e3,
    vorticity.isel(time=0),
    cmap=cmocean.cm.balance,
    vmin=-vmax,
    vmax=vmax,
)
ax.set_aspect("equal")
ax.set_xlabel("x [km]")
ax.set_ylabel("y [km]")
ax.set_title("Relative vorticity")
fig.colorbar(pc, ax=ax, label=r"$\zeta$ [s$^{-1}$]", extend="both")
plt.show()

The vorticity field shows the eddies and meanders of the jet in the channel, with no noise at the grid scale or artefacts at the zonal boundary, which tells us that the staggering of `U` and `V` was correctly accounted for.

## Sampling the vorticity in a Parcels simulation

We add the vorticity as a new variable to the SGRID dataset. Because it is defined on the `(lat, lon)` nodes, which are already described by the SGRID metadata, Parcels knows where it lives on the grid.

In [ ]:
ds_fset["vorticity"] = vorticity
fieldset = parcels.FieldSet.from_sgrid_conventions(ds_fset)
fieldset.vorticity

Following the [sampling tutorial](tutorial_sampling.ipynb), we create a particle class with a `vorticity` variable and a kernel that samples the field at the particle locations.

In [ ]:
VorticityParticle = parcels.Particle.add_variable(
    parcels.Variable("vorticity", dtype=np.float32, initial=np.nan)
)


def SampleVorticity(particles, fieldset):
    particles.vorticity = fieldset.vorticity[
        particles.t, particles.z, particles.y, particles.x
    ]


def DeleteParticle(particles, fieldset):
    any_error = particles.state >= 50
    particles[any_error].state = parcels.StatusCode.Delete

We release a grid of particles at the surface and advect them for 8 days. As in the [MITgcm tutorial](tutorial_mitgcm.ipynb), particles that leave the domain are deleted.

In [ ]:
X, Y = np.meshgrid(np.linspace(100e3, 900e3, 5), np.linspace(200e3, 1800e3, 9))

pset = parcels.ParticleSet(
    fieldset=fieldset,
    pclass=VorticityParticle,
    x=X.ravel(),
    y=Y.ravel(),
    z=np.zeros(X.size),
)

output_file = parcels.ParticleFile(
    path="xgcm_vorticity_particles.parquet",
    outputdt=np.timedelta64(3, "h"),
)

pset.execute(
    [parcels.kernels.AdvectionRK2, SampleVorticity, DeleteParticle],
    runtime=np.timedelta64(8, "D"),
    dt=np.timedelta64(1, "h"),
    output_file=output_file,
)

Finally, we plot the trajectories coloured by the vorticity sampled by the particles, on top of the vorticity field at the start of the simulation.

In [ ]:
df = parcels.read_particlefile("xgcm_vorticity_particles.parquet")

fig, ax = plt.subplots(figsize=(5, 8))
ax.pcolormesh(
    vorticity.lon / 1e3,
    vorticity.lat / 1e3,
    vorticity.isel(time=0),
    cmap=cmocean.cm.balance,
    vmin=-vmax,
    vmax=vmax,
    alpha=0.3,
)
for traj in df.partition_by("particle_id", maintain_order=True):
    ax.plot(traj["x"] / 1e3, traj["y"] / 1e3, c="0.4", lw=0.8, zorder=1)
    sc = ax.scatter(
        traj["x"] / 1e3,
        traj["y"] / 1e3,
        c=traj["vorticity"],
        s=10,
        cmap=cmocean.cm.balance,
        vmin=-vmax,
        vmax=vmax,
        zorder=2,
    )
ax.set_aspect("equal")
ax.set_xlabel("x [km]")
ax.set_ylabel("y [km]")
ax.set_title("Vorticity sampled along trajectories")
fig.colorbar(sc, ax=ax, label=r"$\zeta$ [s$^{-1}$]", extend="both")
plt.show()